In [12]:
# ReAct Prompt Template
REACT_PROMPT_TEMPLATE = """
Please note that you are an intelligent assistant capable of calling external tools.

Available tools are as follows:
{tools}

Current date: {current_date}

For questions asking for the latest or current information:
- Use the current date when forming search queries.
- Prefer authoritative and official sources when available.

Please respond strictly in the following format:

Thought: Your thinking process, used to analyze problems, decompose tasks, and plan the next action.
Action: The action you decide to take, must be in one of the following formats:
- {{tool_name}}[{{tool_input}}]`: Call an available tool.
- `Finish[final answer]`: When you believe you have obtained the final answer.
- When you have collected enough information to answer the user's final question, you must use `Finish[final answer]` after the Action: field to output the final answer.

Now, please start solving the following problem:
Question: {question}
History: {history}
"""


In [16]:
import re
from datetime import datetime
class ReActAgent:
    def __init__(self, llm, tool_executor):
        self.llm = llm
        self.tools = tool_executor

    def run(self,question):
        current_date = datetime.now().strftime("%Y-%m-%d")
        print("1. run() starts")
        steps = 0
        max_step =5
        history = ""

        while True:
            steps +=1
            print("current step:", steps)
            if steps > max_step:
                return None

            tools = self.tools.toolavaliable()
            print("2. tools:", tools)


            prompt = REACT_PROMPT_TEMPLATE.format(
                tools = tools,
                question= question,
                history= history,
                current_date=current_date
            )
            print("3. prompt:")
            print(prompt)

            message = [{
                "role":"user",
                "content":prompt
            }]
            print("4. message ")

            response =self.llm.think(message)
            
            if not response:
                print("LLM did not return valid content.")
                return None

            thought,action = self._parse_output(response)

            print("6. thought:", thought)
            print("7. action:", action)

            if action.startswith("Finish"):
                final_answer = action.split("[")[1].replace("]","")
                return final_answer
            
            tool_name,tool_action = self._parse_action(action)
            print("tool_name:", tool_name)
            print("tool_action:", tool_action)

            tool_func = self.tools.getTool(tool_name)
            print("tool_func:", tool_func)

            print("\n===== TOOL START =====")
            observation = tool_func(tool_action)
            print("\n===== REAL OBSERVATION =====")
            print(observation)

            print("===== TOOL END =====")

            history += f"""
            Thought: {thought}
            Action: {action}
            Observation: {observation}
            """
            
            print("===== CURRENT HISTORY =====")
            print(history)
            


    def _parse_output(self, text: str):
        part = []
        

        for line in text.split("\n"):
            if line.strip():
               part.append(line)

        think_match = re.search(r"Thought:\s*(.*)",text)
        think = think_match.group(1).strip()

        if "Action: Finish" in text:
            action_match = re.search(r"Action:\s*(Finish\[.*?\])", text, re.DOTALL)
        else:
            action_match = re.search(r"Action:\s*(.*)",text)
        action = action_match.group(1).strip()
        

        return think,action

    def _parse_action(self, action_text):
        tool_name = action_text.split("[")[0]

        tool_action = action_text.split("[")[1].replace("]", "")
        return tool_name,tool_action

        

In [18]:
from llm_client import HelloAgentsLLM
from tool_executor import executor

llm_Client = HelloAgentsLLM()
tool_executor = executor

agent = ReActAgent(llm_Client, tool_executor)

result = agent.run(
    "Use the search tool to find the latest information about OpenAI."
)

print("\n===== FINAL RESULT =====")
print(result)


MODEL: nvidia/nemotron-3-ultra-550b-a55b:free
BASE URL: https://openrouter.ai/api/v1
KEY PREFIX: sk-or-v1
1. run() starts
current step: 1
2. tools: -add: add 2 numbers
-search: A web search engine. Use this tool when you need to answer questions about current events, facts, and information not found in your knowledge base.
3. prompt:

Please note that you are an intelligent assistant capable of calling external tools.

Available tools are as follows:
-add: add 2 numbers
-search: A web search engine. Use this tool when you need to answer questions about current events, facts, and information not found in your knowledge base.

Current date: 2026-09-01

For questions asking for the latest or current information:
- Use the current date when forming search queries.
- Prefer authoritative and official sources when available.

Please respond strictly in the following format:

Thought: Your thinking process, used to analyze problems, decompose tasks, and plan the next action.
Action: The actio